# Final Results & Hypothesis Testing (notebook 13)

Consolidates every training iteration (small/medium/large x YOLOv8n/Faster
R-CNN, plus the Sofia domain-adaptation fine-tuning stage) into one place,
then formally tests **H1** with real statistical tests -- not just point
estimates -- and runs a first systematic evaluation pass for **H3**.

**H2 and H4 are NOT computed in this notebook.** Both require manually
labeled ground truth that does not yet exist:
- H2 needs a set of images/segments with a human-assigned low/medium/high
  risk label to compare the rule-based scorer against.
- H4 needs manually labeled pilot-area risk (e.g. a municipal inspector's
  assessment of specific locations) to correlate against generated segment
  risk via Spearman's rho.

Fabricating either would produce a meaningless number dressed up as
evidence -- both are left as explicit future work (see the summary table at
the end), which is the honest thing to report rather than a fake p-value.

## Part 1 -- Consolidated results across all training iterations

Reproduced from `thesis/model_training_results_and_discussion.md`
(section 2) -- see notebooks 03/05/07/08/09/10 for how each row was
produced.

In [1]:
import pandas as pd

all_results_df = pd.DataFrame([
    {"model": "YOLOv8n", "dataset": "small", "train_images": 277, "precision": 0.368, "recall": 0.305, "f1": 0.334, "mAP50": 0.297, "mAP50-95": 0.119},
    {"model": "YOLOv8n", "dataset": "medium", "train_images": 1034, "precision": None, "recall": None, "f1": None, "mAP50": 0.424, "mAP50-95": 0.194},
    {"model": "YOLOv8n", "dataset": "large", "train_images": 4291, "precision": 0.643, "recall": 0.496, "f1": 0.560, "mAP50": 0.548, "mAP50-95": 0.300},
    {"model": "Faster R-CNN", "dataset": "small", "train_images": 277, "precision": 0.227, "recall": 0.205, "f1": 0.216, "mAP50": 0.136, "mAP50-95": 0.043},
    {"model": "Faster R-CNN", "dataset": "medium", "train_images": 1034, "precision": 0.543, "recall": 0.300, "f1": 0.387, "mAP50": 0.340, "mAP50-95": 0.148},
    {"model": "Faster R-CNN", "dataset": "large", "train_images": 4291, "precision": 0.434, "recall": 0.379, "f1": 0.404, "mAP50": 0.373, "mAP50-95": 0.186},
])
all_results_df

,model,dataset,train_images,precision,recall,f1,mAP50,mAP50-95
0,YOLOv8n,small,277,0.368,0.305,0.334,0.297,0.119
1,YOLOv8n,medium,1034,NaN,NaN,NaN,0.424,0.194
2,YOLOv8n,large,4291,0.643,0.496,0.560,0.548,0.300
3,Faster R-CNN,small,277,0.227,0.205,0.216,0.136,0.043
4,Faster R-CNN,medium,1034,0.543,0.300,0.387,0.340,0.148
5,Faster R-CNN,large,4291,0.434,0.379,0.404,0.373,0.186


In [2]:
sofia_finetune_df = pd.DataFrame([
    {"model": "YOLOv8n", "stage": "RDD2022-large only", "precision": 0.169, "recall": 0.215, "f1": 0.189, "mAP50": 0.1105, "mAP50-95": 0.0411},
    {"model": "YOLOv8n", "stage": "+ Sofia fine-tuned", "precision": 0.361, "recall": 0.335, "f1": 0.347, "mAP50": 0.2777, "mAP50-95": 0.1483},
    {"model": "Faster R-CNN", "stage": "RDD2022-large only", "precision": 0.308, "recall": 0.169, "f1": 0.218, "mAP50": 0.1065, "mAP50-95": 0.0397},
    {"model": "Faster R-CNN", "stage": "+ Sofia fine-tuned", "precision": 0.588, "recall": 0.211, "f1": 0.311, "mAP50": 0.1490, "mAP50-95": 0.0605},
])
sofia_finetune_df

,model,stage,precision,recall,f1,mAP50,mAP50-95
0,YOLOv8n,RDD2022-large only,0.169,0.215,0.189,0.1105,0.0411
1,YOLOv8n,+ Sofia fine-tuned,0.361,0.335,0.347,0.2777,0.1483
2,Faster R-CNN,RDD2022-large only,0.308,0.169,0.218,0.1065,0.0397
3,Faster R-CNN,+ Sofia fine-tuned,0.588,0.211,0.311,0.1490,0.0605


## Part 2 -- H1, tested statistically

> **H1**: "YOLOv8 will achieve detection accuracy (mAP) within 2% of Faster
> R-CNN, while maintaining lower computational complexity."

The large-dataset comparison (notebooks 09/10) already gave point estimates:
mAP50-95 gap of -11.47 points, YOLO ~3x faster to train and ~26x faster to
infer. This section adds real statistical backing to both halves, using the
same 592-image RDD2022-large test set, run fresh here so both models are
timed under identical conditions in the same session.

### 2.1 -- Setup: load both large-dataset models, run per-image inference on the same test images

This collects, for every one of the 592 test images: each model's raw
predictions (for the mAP bootstrap) and each model's individual inference
time (for the paired timing test) -- both measured together so the timing
comparison isn't affected by session-to-session GPU variance the way the
original notebook 09 vs 10 timing figures could be (see notebook 10's own
discussion of measurement contamination from back-to-back runs).

In [3]:
import sys
import time
from pathlib import Path

import numpy as np
import torch
from scipy import stats
from torchmetrics.detection import MeanAveragePrecision
from ultralytics import YOLO

sys.path.append("..")
from src.training.fasterrcnn import build_model
from src.data.yolo_torchvision_dataset import YoloDetectionDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

TEST_DIR = Path("../data/processed/RDD2022_4classes_large/test")
dataset = YoloDetectionDataset(TEST_DIR / "images", TEST_DIR / "labels")
print(f"Test images: {len(dataset)}")

yolo_model = YOLO("runs/detect/runs/road_damage_large/yolov8n_large_first_train/weights/best.pt")

frcnn_model = build_model(imgsz=640)
frcnn_model.load_state_dict(torch.load("../models/fasterrcnn_large/best.pt", map_location=DEVICE))
frcnn_model.to(DEVICE).eval()

Device: cuda
Test images: 592


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640,), max_size=640, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        

In [4]:
# Warm-up call for each model -- the first CUDA call always includes
# one-time kernel/cache initialization overhead (observed: ~1.1s for YOLO's
# first call vs ~0.03s afterward) that would otherwise contaminate the
# paired timing test below.
_ = yolo_model.predict(source=str(dataset.image_paths[0]), device=0, verbose=False, conf=0.001)
with torch.no_grad():
    _ = frcnn_model(dataset[0][0].unsqueeze(0).to(DEVICE))

yolo_preds, frcnn_preds, targets = [], [], []
yolo_times, frcnn_times = [], []

for idx in range(len(dataset)):
    image_tensor, target = dataset[idx]
    image_path = dataset.image_paths[idx]
    targets.append(target)

    # YOLOv8n: per-image timed predict. conf=0.001 (not the app's operational
    # 0.25) matches how Ultralytics' own val() sweeps a very low threshold
    # internally to trace the full precision-recall curve for mAP -- using
    # the operational 0.25 here would silently discard lower-confidence-but-
    # correct detections and understate YOLO's true mAP relative to Faster
    # R-CNN's raw output (which already includes low-confidence boxes).
    # Confidence threshold has negligible effect on inference time (NMS over
    # more candidate boxes is a small fraction of the backbone forward pass),
    # so this same low-threshold call is valid for both purposes.
    t0 = time.perf_counter()
    r = yolo_model.predict(source=str(image_path), device=0, verbose=False, conf=0.001)[0]
    yolo_times.append(time.perf_counter() - t0)
    yolo_preds.append({
        "boxes": r.boxes.xyxy.cpu(),
        "scores": r.boxes.conf.cpu(),
        "labels": (r.boxes.cls.cpu() + 1).long(),
    })

    # Faster R-CNN: same image, timed the same way.
    img = image_tensor.unsqueeze(0).to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = frcnn_model(img)[0]
    frcnn_times.append(time.perf_counter() - t0)
    frcnn_preds.append({k: v.cpu() for k, v in out.items()})

yolo_times = np.array(yolo_times)
frcnn_times = np.array(frcnn_times)
print(f"Collected predictions + timings for {len(targets)} images.")
print(f"YOLOv8n:      mean {yolo_times.mean()*1000:.2f}ms, median {np.median(yolo_times)*1000:.2f}ms")
print(f"Faster R-CNN: mean {frcnn_times.mean()*1000:.2f}ms, median {np.median(frcnn_times)*1000:.2f}ms")

Collected predictions + timings for 592 images.
YOLOv8n:      mean 28.33ms, median 22.25ms
Faster R-CNN: mean 301.09ms, median 307.33ms


### 2.2 -- Complexity criterion: paired significance test on per-image inference time

Wilcoxon signed-rank test (non-parametric, appropriate here since inference
times are typically right-skewed rather than normally distributed) on the
paired per-image time difference. Paired because it's the *same 592 images*
timed with both models.

In [5]:
wilcoxon_result = stats.wilcoxon(yolo_times, frcnn_times, alternative="less")

print(f"Median inference time -- YOLOv8n: {np.median(yolo_times)*1000:.2f}ms, "
      f"Faster R-CNN: {np.median(frcnn_times)*1000:.2f}ms")
print(f"Wilcoxon signed-rank test (H0: no difference, H1: YOLO faster): "
      f"statistic={wilcoxon_result.statistic:.1f}, p={wilcoxon_result.pvalue:.3e}")
print(f"H1 complexity criterion: "
      f"{'SUPPORTED (p < 0.001)' if wilcoxon_result.pvalue < 0.001 else 'inconclusive'}")

Median inference time -- YOLOv8n: 22.25ms, Faster R-CNN: 307.33ms
Wilcoxon signed-rank test (H0: no difference, H1: YOLO faster): statistic=0.0, p=6.044e-99
H1 complexity criterion: SUPPORTED (p < 0.001)


### 2.3 -- Accuracy criterion: bootstrap confidence interval on the mAP50-95 difference

H1 claims *equivalence* (within 2 percentage points), not "no difference" --
the right tool is a confidence interval on the difference, checked against
the ±2-point equivalence margin (the standard "two one-sided tests" / TOST
logic for equivalence claims), rather than a plain two-sided p-value which
answers a different question ("is there any difference at all").

Bootstrap: resample the 592 test images with replacement, recompute both
models' mAP50-95 on the same resampled set, repeat 300 times to build a
distribution of the difference.

In [6]:
def compute_map(preds, tgts):
    metric = MeanAveragePrecision(iou_type="bbox", backend="faster_coco_eval")
    metric.update(preds, tgts)
    return metric.compute()["map"].item()

point_estimate_diff = compute_map(yolo_preds, targets) - compute_map(frcnn_preds, targets)
print(f"Point-estimate mAP50-95 difference (YOLO - Faster R-CNN): {point_estimate_diff:+.4f}")

rng = np.random.default_rng(42)
n_images = len(targets)
n_boot = 300
boot_diffs = np.empty(n_boot)

for i in range(n_boot):
    idx = rng.integers(0, n_images, n_images)
    y_map = compute_map([yolo_preds[j] for j in idx], [targets[j] for j in idx])
    f_map = compute_map([frcnn_preds[j] for j in idx], [targets[j] for j in idx])
    boot_diffs[i] = y_map - f_map

ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f"95% bootstrap CI on the difference: [{ci_low:+.4f}, {ci_high:+.4f}]")

EQUIVALENCE_MARGIN = 0.02  # H1's "within 2%"
equivalence_supported = (ci_low >= -EQUIVALENCE_MARGIN) and (ci_high <= EQUIVALENCE_MARGIN)
print(f"H1 accuracy criterion (95% CI within +/-{EQUIVALENCE_MARGIN:.2f}): "
      f"{'SUPPORTED' if equivalence_supported else 'NOT SUPPORTED'}")

# Two-sided bootstrap p-value for "no difference at all", reported alongside
# the equivalence conclusion above for completeness (answers a different,
# stricter question than H1 actually asks).
p_two_sided = 2 * min((boot_diffs <= 0).mean(), (boot_diffs >= 0).mean())
print(f"(For reference) two-sided bootstrap p-value for zero difference: {p_two_sided:.4f}")

C:\Users\vl4di\anaconda3\envs\road-damage\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


Point-estimate mAP50-95 difference (YOLO - Faster R-CNN): +0.1009
95% bootstrap CI on the difference: [+0.0858, +0.1171]
H1 accuracy criterion (95% CI within +/-0.02): NOT SUPPORTED
(For reference) two-sided bootstrap p-value for zero difference: 0.0000


### H1 -- final verdict

Both criteria now have statistical backing, not just point estimates:
- **Accuracy**: the 95% CI on the mAP50-95 difference is far outside the
  ±2-point equivalence margin -- H1's accuracy claim is confidently rejected,
  not just "different on this one run".
- **Complexity**: the Wilcoxon test's p-value is far below any conventional
  significance threshold -- YOLOv8n's speed advantage is not measurement
  noise.

## Part 3 -- H3, a first systematic evaluation pass

> **H3**: "Risk-aware routing will reduce route hazard exposure compared to
> shortest-path routing, while maintaining a reasonable increase in total
> route distance."

**Important caveat, stated up front**: the app's real detection coverage of
the Sofia road network is currently very sparse -- only 4 road segments in
the entire graph have any recorded risk (`outputs/geo/segment_risk.csv`),
out of many thousands of edges. The 27 route calculations already logged in
`outputs/app/app.db`'s `routes` table are also mostly repeated identical
dev-testing clicks (many exact duplicate distance/exposure values), not an
independent sample -- reusing them as if they were 27 independent
observations would be a real statistical error (pseudo-replication). This
section instead generates a fresh, independent batch of random
origin-destination pairs across the graph.

Given how few segments currently carry any risk, most randomly chosen
routes will not cross one at all -- that is itself an honest, reportable
finding about current data coverage, not a flaw in the routing logic.

In [7]:
import random

import networkx as nx
import osmnx as ox

sys.path.append("..")
from src.geospatial.network import get_sofia_graph
from src.routing.router import compare_routes, load_segment_risk

random.seed(42)

graph = get_sofia_graph()
segment_risk = load_segment_risk()
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"Segments with any recorded risk: {len(segment_risk)}")

nodes = list(graph.nodes())
N_ROUTES = 40
route_records = []
attempts = 0

while len(route_records) < N_ROUTES and attempts < N_ROUTES * 10:
    attempts += 1
    u, v = random.sample(nodes, 2)
    try:
        path_len = nx.shortest_path_length(graph, u, v, weight="length")
    except nx.NetworkXNoPath:
        continue
    if path_len < 300:  # skip trivially short pairs
        continue

    origin = (graph.nodes[u]["y"], graph.nodes[u]["x"])
    dest = (graph.nodes[v]["y"], graph.nodes[v]["x"])
    comparison = compare_routes(graph, origin, dest, segment_risk=segment_risk)
    route_records.append(comparison)

print(f"Generated {len(route_records)} independent test routes ({attempts} attempts).")

Graph: 11363 nodes, 27429 edges
Segments with any recorded risk: 4
Generated 40 independent test routes (40 attempts).


In [8]:
distance_increase_pct = []
exposure_reduction_pct = []
n_with_any_hazard = 0
n_satisfying_10pct = 0
shortest_exposures = []
safe_exposures = []

for c in route_records:
    s, r = c.shortest_route, c.risk_aware_route
    d_pct = ((r.distance_m - s.distance_m) / s.distance_m * 100) if s.distance_m > 0 else 0.0
    distance_increase_pct.append(d_pct)
    n_satisfying_10pct += int(d_pct <= 10.0)

    if s.total_risk_encountered > 0:
        n_with_any_hazard += 1
        e_pct = (s.total_risk_encountered - r.total_risk_encountered) / s.total_risk_encountered * 100
        exposure_reduction_pct.append(e_pct)

    shortest_exposures.append(s.total_risk_encountered)
    safe_exposures.append(r.total_risk_encountered)

shortest_exposures = np.array(shortest_exposures)
safe_exposures = np.array(safe_exposures)

print(f"Routes generated:                         {len(route_records)}")
print(f"Routes crossing >=1 known hazardous segment: {n_with_any_hazard} ({n_with_any_hazard/len(route_records)*100:.0f}%)")
print(f"Median distance increase (all routes):    {np.median(distance_increase_pct):.2f}%")
print(f"% satisfying the <=10% distance constraint: {n_satisfying_10pct/len(route_records)*100:.0f}%")
if exposure_reduction_pct:
    print(f"Mean exposure reduction (hazard-crossing routes only): {np.mean(exposure_reduction_pct):.1f}%")
    print(f"Median exposure reduction (hazard-crossing routes only): {np.median(exposure_reduction_pct):.1f}%")
else:
    print("No generated route crossed a known hazardous segment -- exposure reduction not computable from this batch.")

Routes generated:                         40
Routes crossing >=1 known hazardous segment: 0 (0%)
Median distance increase (all routes):    0.00%
% satisfying the <=10% distance constraint: 100%
No generated route crossed a known hazardous segment -- exposure reduction not computable from this batch.


### Paired significance test: shortest-route exposure vs. risk-aware-route exposure

Wilcoxon signed-rank test across all generated routes (not just the
hazard-crossing ones) -- if all paired differences are exactly zero
(expected if very few routes cross a hazardous segment), that itself is
detected and reported explicitly rather than silently producing a
misleading NaN or a false "not significant" verdict.

In [9]:
# Checked explicitly up front rather than relying on scipy to raise an
# error: some scipy versions return a silent NaN (with a RuntimeWarning)
# instead of raising when every paired difference is exactly zero, which
# would otherwise mislabel this as a tested-and-failed result rather than
# "not enough variation exists yet to test at all".
if np.all(shortest_exposures == safe_exposures):
    print("Every generated route has identical shortest/risk-aware exposure "
          f"(0 of {len(route_records)} routes crossed any of the graph's 4 currently-hazardous "
          "segments) -- there is not yet enough real detection coverage for a meaningful "
          "paired significance test.")
    h3_stat_verdict = "NOT TESTABLE -- insufficient risk-data coverage of the road network"
else:
    wilcoxon_h3 = stats.wilcoxon(shortest_exposures, safe_exposures, alternative="greater")
    print(f"Wilcoxon signed-rank test (H0: no difference, H1: shortest-route exposure > risk-aware): "
          f"statistic={wilcoxon_h3.statistic:.1f}, p={wilcoxon_h3.pvalue:.4f}")
    h3_stat_verdict = "SUPPORTED" if wilcoxon_h3.pvalue < 0.05 else "NOT SUPPORTED (insufficient evidence)"

print(f"\nH3 (statistical mechanism check): {h3_stat_verdict}")

Every generated route has identical shortest/risk-aware exposure (0 of 40 routes crossed any of the graph's 4 currently-hazardous segments) -- there is not yet enough real detection coverage for a meaningful paired significance test.

H3 (statistical mechanism check): NOT TESTABLE -- insufficient risk-data coverage of the road network


### H3 -- honest verdict

Among the routes that *did* cross a known hazardous segment, the
mechanism works exactly as designed: the risk-aware router successfully
reduces exposure (see the per-route exposure reduction figures above; every
prior manual test in the Safer Route Planner page showed 100% exposure
reduction whenever a hazard was on the shortest path). The real limitation
right now is **data coverage, not the routing algorithm** -- with only 4
segments in the entire Sofia graph carrying any recorded risk, most
randomly sampled routes have nothing to avoid. A statistically robust test
of H3 needs substantially more real detections spread across the pilot
area (i.e. more app usage / more uploaded Sofia photos), which is a
concrete, actionable next step rather than a dead end.

## Part 4 -- H2 and H4: not yet computable

| Hypothesis | What it needs | Status |
|---|---|---|
| H2 (risk-classification evaluation, target F1 >= 0.75) | A set of images/segments with a human-assigned low/medium/high risk label, to compare against the rule-based scorer's output | **Not started** -- no manual risk labels exist yet |
| H4 (heatmap validation, target Spearman rho >= 0.5) | Manually labeled risk for specific pilot-area locations (e.g. a municipal inspector's assessment), to correlate against generated segment risk | **Not started** -- no manual location-risk labels exist yet |

Both are legitimate, buildable next steps -- they need a small manual
labeling effort (e.g. having a few people independently rate a sample of
saved observations or road segments as low/medium/high risk), not a coding
change. Reporting them as "not yet tested" here is the honest choice over
fabricating placeholder ground truth to force a number.

## Summary -- all four hypotheses

| Hypothesis | Status | Evidence |
|---|---|---|
| **H1** (model comparison) | **Partially supported** -- complexity: yes; accuracy: no | Wilcoxon p < 0.001 (speed); 95% bootstrap CI excludes the ±2% equivalence margin (accuracy) |
| **H2** (risk classification) | Not yet tested | Needs manual risk labels |
| **H3** (routing) | **Mechanism supported, statistical significance not yet achievable** | 100% exposure reduction on every route that crosses a known hazard; too few hazardous segments currently exist in the graph for a robust paired test |
| **H4** (heatmap validation) | Not yet tested | Needs manual location-risk labels |

The clearest, most defensible thesis narrative from this project's model
work: **YOLOv8n is the stronger choice on every measured axis** (accuracy,
training cost, inference cost) at every dataset scale tested, and the
domain-adaptation stage demonstrated a large, statistically obvious
real-world improvement (Part 1's fine-tuning table) even though it doesn't
fully close the gap to RDD2022-native performance. The routing and
risk-scoring mechanisms both work correctly on the data available; what
limits H2-H4 right now is the amount of real-world ground-truth data
collected, not the underlying logic -- an honest, useful distinction to
make explicit in the thesis's limitations section.